# Model comparison and detectability analysis

This notebook is part of the reproducibility material associated with the manuscript:

**Detectability limits of tree-covered coffee at Sentinel resolution: asymmetric gains from optical texture**

## Purpose

This notebook integrates the derived predictors from the Sentinel-1 SAR and Sentinel-2 optical workflows and compares the modelling scenarios reported in the manuscript:

1. **SAR** — Sentinel-1 C-band structural and temporal metrics.
2. **Optical** — Sentinel-2 spectral bands and vegetation indices.
3. **Optical–SAR Fusion** — combined Sentinel-2 optical and Sentinel-1 SAR predictors.
4. **Fusion + Sentinel-2 GLCM** — combined optical, SAR and optical texture predictors.

The notebook focuses on sample-level detectability, probabilistic separability, feature importance and directional classification errors between tree-covered coffee and forest.

## Scientific rationale

The central assumption is that tree-covered coffee may not behave as a uniformly observable land-cover class at Sentinel resolution. Instead, its spectral, structural and textural response can converge with forest formations. For this reason, the workflow evaluates not only global classification metrics, but also probability distributions, shade-level behaviour and asymmetric error transitions between scenarios.

## Data privacy and reproducibility note

The notebook expects previously derived, sample-level CSV tables. These tables should not contain polygon geometries, producer names, precise farm identifiers or other restricted information when used in a public repository. The original geospatial reference polygons should remain private unless explicit data-sharing authorisation exists.

Only derived, anonymised and non-geometric tables should be exported or shared publicly.


## 1. Imports and configuration

This section defines the computational environment, cross-validation settings and input/output paths. The notebook is designed to run in Google Colab or a local Jupyter environment, provided that the CSV files exported by the previous notebooks are available.


In [ ]:
# ============================================================
# BLOCK 1 — Imports, modelling settings and display options
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.inspection import permutation_importance

from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
N_SPLITS = 5
N_ESTIMATORS = 300

print("Libraries imported successfully.")

In [ ]:
# ============================================================
# BLOCK 2 — User configuration
# ============================================================

# Optional for Google Colab users:
# Uncomment the following lines only when the input CSV files are stored in Google Drive.
#
# from google.colab import drive
# drive.mount('/content/drive')

# Set this directory to the folder containing the CSV files exported by the
# Sentinel-1 and Sentinel-2 notebooks. Avoid committing private paths, raw
# geospatial data or credentials to a public repository.
PROJECT_DIR = "path/to/project_directory"

# Main CSV files exported from the sensor-specific notebooks.
s1_csv_path = os.path.join(PROJECT_DIR, "SAR_features_DJFM_22_25_main_noDEM.csv")
s2_csv_path = os.path.join(PROJECT_DIR, "S2_optical_attributes_reduced_DJFM_3class.csv")

# Optional Sentinel-2 GLCM table exported from the Sentinel-2 notebook.
# If this file is unavailable, the Fusion + S2 GLCM scenario will be skipped.
s2_glcm_csv_path = os.path.join(PROJECT_DIR, "S2_GLCM_texture_DJFM_3class_optional.csv")

# Output directory for derived, non-geometric tables.
output_dir = os.path.join(PROJECT_DIR, "integration_outputs")

print("S1 CSV path:", s1_csv_path)
print("S2 CSV path:", s2_csv_path)
print("S2 GLCM CSV path:", s2_glcm_csv_path)
print("Output directory:", output_dir)


## 2. Read and standardise input tables

The analysis uses the CSV outputs from the Sentinel-1 and Sentinel-2 notebooks. The table structure is standardised around a common `sample_id` field and a binary `class` label (`coffee` or `forest`). When available, `New_ID` is retained to support shade-level interpretation within coffee polygons.


In [ ]:
# ============================================================
# BLOCK 3 — Read input tables
# ============================================================

def read_csv_checked(path, required=True):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Loaded: {path}")
        print("Shape:", df.shape)
        return df
    if required:
        raise FileNotFoundError(f"Required file not found: {path}")
    print(f"[WARNING] Optional file not found: {path}")
    return None

s1_df = read_csv_checked(s1_csv_path, required=True)
s2_df = read_csv_checked(s2_csv_path, required=True)
s2_glcm_df = read_csv_checked(s2_glcm_csv_path, required=False)

print("\nS1 columns:")
print(list(s1_df.columns))

print("\nS2 columns:")
print(list(s2_df.columns))

if s2_glcm_df is not None:
    print("\nS2 GLCM columns:")
    print(list(s2_glcm_df.columns))

In [ ]:
# ============================================================
# BLOCK 4 — Standardise key columns
# ============================================================

def standardise_base_columns(df, name):
    df = df.copy()

    # Standardise class field
    possible_class_cols = ["class", "Class", "CLASS"]
    class_col = None
    for c in possible_class_cols:
        if c in df.columns:
            class_col = c
            break
    if class_col is None:
        raise ValueError(f"No class column found in {name}. Expected one of: {possible_class_cols}")

    if class_col != "class":
        df = df.rename(columns={class_col: "class"})

    df["class"] = df["class"].astype(str).str.strip().str.lower()

    # Standardise sample_id
    if "sample_id" not in df.columns:
        raise ValueError(f"sample_id not found in {name}.")
    df["sample_id"] = df["sample_id"].astype(str)

    # Standardise New_ID when available
    if "New_ID" in df.columns:
        df["New_ID"] = pd.to_numeric(df["New_ID"], errors="coerce")
    elif "new_id" in df.columns:
        df = df.rename(columns={"new_id": "New_ID"})
        df["New_ID"] = pd.to_numeric(df["New_ID"], errors="coerce")

    return df

s1_df = standardise_base_columns(s1_df, "S1")
s2_df = standardise_base_columns(s2_df, "S2")

if s2_glcm_df is not None:
    s2_glcm_df = standardise_base_columns(s2_glcm_df, "S2 GLCM")

print("Base columns standardised.")

print("\nS1 class distribution:")
print(s1_df["class"].value_counts(dropna=False))

print("\nS2 class distribution:")
print(s2_df["class"].value_counts(dropna=False))

if s2_glcm_df is not None:
    print("\nS2 GLCM class distribution:")
    print(s2_glcm_df["class"].value_counts(dropna=False))

## 3. Predictor groups

The feature groups below define the exact variables used in each modelling scenario. The notebook intentionally excludes DEM, climate and ancillary topographic variables so that the comparison focuses on Sentinel-1 SAR, Sentinel-2 optical information and Sentinel-2 optical texture.


In [ ]:
# ============================================================
# BLOCK 5 — Define final feature sets
# ============================================================

OPTICAL_FEATURES = [
    "B2_median",
    "B3_median",
    "B4_median",
    "B5_median",
    "B6_median",
    "B7_median",
    "B8_median",
    "B8A_median",
    "NDVI_median",
    "NDRE_median",
    "MCARI2_median",
    "NIRv_median"
]

SAR_FEATURES = [
    "DJFM_22_25_VV_med",
    "DJFM_22_25_VH_med",
    "DJFM_22_25_ratio_med",
    "DJFM_22_25_ratio_sd",
    "DJFM_22_25_ratio_p90_minus_p10"
]

S2_GLCM_FEATURES = [
    "B4_glcm_contrast",
    "B4_glcm_entropy",
    "NDRE_glcm_contrast",
    "NDRE_glcm_entropy"
]

print("Optical features:", len(OPTICAL_FEATURES))
print(OPTICAL_FEATURES)

print("\nSAR features:", len(SAR_FEATURES))
print(SAR_FEATURES)

print("\nS2 GLCM features:", len(S2_GLCM_FEATURES))
print(S2_GLCM_FEATURES)

In [ ]:
# ============================================================
# BLOCK 6 — Check feature availability
# ============================================================

def check_features(df, features, df_name):
    missing = [f for f in features if f not in df.columns]
    available = [f for f in features if f in df.columns]
    print(f"\n{df_name}: {len(available)}/{len(features)} expected features available.")
    if missing:
        print("[WARNING] Missing features:")
        for f in missing:
            print("-", f)
    return available, missing

available_optical, missing_optical = check_features(s2_df, OPTICAL_FEATURES, "S2 optical table")
available_sar, missing_sar = check_features(s1_df, SAR_FEATURES, "S1 SAR table")

if s2_glcm_df is not None:
    available_glcm, missing_glcm = check_features(s2_glcm_df, S2_GLCM_FEATURES, "S2 GLCM table")
else:
    available_glcm, missing_glcm = [], S2_GLCM_FEATURES

if missing_optical:
    raise ValueError("Missing required Optical features. Check the S2 CSV.")
if missing_sar:
    raise ValueError("Missing required SAR features. Check the S1 CSV.")

print("\nRequired Optical and SAR features are available.")

## 4. Prepare scenario-specific modelling tables

The tables are filtered to the binary coffee–forest problem and then joined by `sample_id`. This ensures that each scenario uses only samples with the required predictor set available.


In [ ]:
# ============================================================
# BLOCK 7 — Prepare Optical and SAR dataframes
# ============================================================

# Keep only binary classes
valid_classes = ["coffee", "forest"]

s2_bin = s2_df[s2_df["class"].isin(valid_classes)].copy()
s1_bin = s1_df[s1_df["class"].isin(valid_classes)].copy()

# Keep relevant metadata
meta_cols = ["sample_id", "class"]
if "New_ID" in s2_bin.columns:
    meta_cols.append("New_ID")

s2_model_df = s2_bin[meta_cols + OPTICAL_FEATURES].copy()

s1_meta_cols = ["sample_id", "class"]
if "New_ID" in s1_bin.columns:
    s1_meta_cols.append("New_ID")

s1_model_df = s1_bin[s1_meta_cols + SAR_FEATURES].copy()

print("S2 model dataframe shape:", s2_model_df.shape)
print("S1 model dataframe shape:", s1_model_df.shape)

print("\nS2 class counts:")
print(s2_model_df["class"].value_counts())

print("\nS1 class counts:")
print(s1_model_df["class"].value_counts())

In [ ]:
# ============================================================
# BLOCK 8 — Merge Optical and SAR tables
# ============================================================

# Use S2 as the reference table for class and New_ID.
fusion_df = s2_model_df.merge(
    s1_model_df[["sample_id"] + SAR_FEATURES],
    on="sample_id",
    how="inner"
)

print("Fusion dataframe shape:", fusion_df.shape)
print("\nFusion class counts:")
print(fusion_df["class"].value_counts())

# Check if any rows were lost during merge
lost_from_s2 = set(s2_model_df["sample_id"]) - set(fusion_df["sample_id"])
lost_from_s1 = set(s1_model_df["sample_id"]) - set(fusion_df["sample_id"])

print("\nRows lost from S2 after merge:", len(lost_from_s2))
print("Rows lost from S1 after merge:", len(lost_from_s1))

In [ ]:
# ============================================================
# BLOCK 9 — Merge optional S2 GLCM table
# ============================================================

fusion_glcm_df = None

if s2_glcm_df is not None and len(available_glcm) == len(S2_GLCM_FEATURES):
    glcm_cols = ["sample_id"] + S2_GLCM_FEATURES
    fusion_glcm_df = fusion_df.merge(
        s2_glcm_df[glcm_cols],
        on="sample_id",
        how="inner"
    )

    print("Fusion + S2 GLCM dataframe shape:", fusion_glcm_df.shape)

    lost_from_fusion = set(fusion_df["sample_id"]) - set(fusion_glcm_df["sample_id"])
    print("Rows lost from Fusion after adding GLCM:", len(lost_from_fusion))
else:
    print("[WARNING] S2 GLCM table not available or incomplete.")
    print("Fusion + S2 GLCM scenario will be skipped.")

## 5. Modelling functions and diagnostic summaries

Random Forest models are evaluated using stratified five-fold cross-validation and out-of-fold predictions. This structure avoids reporting predictions from models evaluated on the same samples used for training within each fold. The diagnostic summaries include confusion matrices, feature importance, directional errors and shade-level summaries.


In [ ]:
# ============================================================
# BLOCK 10 — Helper functions for modelling and diagnostics
# ============================================================

def add_group_column(df):
    df = df.copy()

    def assign_group(row):
        if row["class"] == "forest":
            return "forest"
        if row["class"] == "coffee":
            if "New_ID" not in df.columns or pd.isna(row.get("New_ID", np.nan)):
                return "coffee_unknown_shade"
            if int(row["New_ID"]) == 1:
                return "coffee_low_shade"
            if int(row["New_ID"]) == 2:
                return "coffee_intermediate_shade"
            if int(row["New_ID"]) == 3:
                return "coffee_high_shade"
            return "coffee_unknown_shade"
        return "unknown"

    df["group"] = df.apply(assign_group, axis=1)
    return df


def run_rf_cv(df, features, scenario_name):
    data = df.copy()
    data = add_group_column(data)

    # Drop rows with missing feature values
    before = data.shape[0]
    data = data.dropna(subset=features + ["class"]).copy()
    after = data.shape[0]

    if before != after:
        print(f"[{scenario_name}] Rows dropped due to missing values: {before - after}")

    X = data[features].values
    y = data["class"].map({"forest": 0, "coffee": 1}).values

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    model = RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )

    y_pred = cross_val_predict(model, X, y, cv=cv, method="predict")
    y_prob = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]

    metrics = {
        "Scenario": scenario_name,
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall": recall_score(y, y_pred, zero_division=0),
        "F1-score": f1_score(y, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y, y_prob),
        "Average precision": average_precision_score(y, y_prob),
        "n": len(y),
        "n_features": len(features)
    }

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    cm_df = pd.DataFrame(
        cm,
        index=["True Forest", "True Coffee"],
        columns=["Predicted Forest", "Predicted Coffee"]
    )

    results = data[["sample_id", "class", "group"]].copy()
    if "New_ID" in data.columns:
        results["New_ID"] = data["New_ID"].values

    results["y_true"] = y
    results["y_pred"] = y_pred
    results["predicted_class"] = np.where(y_pred == 1, "coffee", "forest")
    results["y_prob_coffee"] = y_prob
    results["correct"] = results["y_true"] == results["y_pred"]
    results["scenario"] = scenario_name

    # Fit model on all data only for feature importance interpretation
    final_model = RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )
    final_model.fit(X, y)

    importance_df = pd.DataFrame({
        "Feature": features,
        "Importance": final_model.feature_importances_
    }).sort_values("Importance", ascending=False).reset_index(drop=True)

    return {
        "scenario": scenario_name,
        "metrics": metrics,
        "cm": cm_df,
        "results": results,
        "importance": importance_df,
        "model": final_model,
        "features": features,
        "data": data
    }


def summarise_error_direction(results):
    df = results.copy()

    def classify_error_direction(row):
        if row["correct"]:
            return "Correct"
        if row["class"] == "coffee" and row["predicted_class"] == "forest":
            return "Coffee_to_Forest"
        if row["class"] == "forest" and row["predicted_class"] == "coffee":
            return "Forest_to_Coffee"
        return "Other"

    df["error_direction"] = df.apply(classify_error_direction, axis=1)

    summary = (
        df.groupby("error_direction")
        .agg(
            n=("sample_id", "count"),
            mean_prob_coffee=("y_prob_coffee", "mean"),
            sd_prob_coffee=("y_prob_coffee", "std")
        )
        .reset_index()
        .sort_values("error_direction")
    )

    return summary


def summarise_shade(results):
    coffee = results[results["class"] == "coffee"].copy()

    if coffee.empty:
        return pd.DataFrame()

    summary = (
        coffee.groupby("group")
        .agg(
            n=("sample_id", "count"),
            correct=("correct", "sum"),
            accuracy=("correct", "mean"),
            mean_prob_coffee=("y_prob_coffee", "mean"),
            sd_prob_coffee=("y_prob_coffee", "std"),
            min_prob_coffee=("y_prob_coffee", "min"),
            max_prob_coffee=("y_prob_coffee", "max")
        )
        .reset_index()
    )

    summary["misclassified"] = summary["n"] - summary["correct"]
    summary["error_rate"] = summary["misclassified"] / summary["n"]

    shade_order = ["coffee_low_shade", "coffee_intermediate_shade", "coffee_high_shade", "coffee_unknown_shade"]
    summary["group"] = pd.Categorical(summary["group"], categories=shade_order, ordered=True)
    summary = summary.sort_values("group")

    return summary


def print_scenario_summary(result):
    scenario = result["scenario"]
    metrics = result["metrics"]

    print("\n" + "="*80)
    print(f"{scenario} — FINAL SUMMARY")
    print("="*80)

    print(f"\n{scenario} MODEL:")
    print(f"Accuracy = {metrics['Accuracy']:.4f}")
    print(f"Precision = {metrics['Precision']:.4f}")
    print(f"Recall = {metrics['Recall']:.4f}")
    print(f"F1-score = {metrics['F1-score']:.4f}")
    print(f"ROC AUC = {metrics['ROC AUC']:.4f}")
    print(f"Average precision = {metrics['Average precision']:.4f}")
    print(f"n = {metrics['n']}")
    print(f"n_features = {metrics['n_features']}")

    print("\nCONFUSION MATRIX:")
    display(result["cm"])

    print("\nFEATURE IMPORTANCE:")
    display(result["importance"])

    print("\nERROR DIRECTION SUMMARY:")
    display(summarise_error_direction(result["results"]))

    print("\nSHADE-LEVEL ERROR SUMMARY:")
    display(summarise_shade(result["results"]))

    print("\nFEATURES USED:")
    for f in result["features"]:
        print("-", f)

    print("\n" + "="*80)
    print(f"END OF {scenario} SUMMARY")
    print("="*80)

## 6. Run the modelling scenarios

The four scenarios are run using the same cross-validation design and model settings, allowing direct comparison of performance and error behaviour.


In [ ]:
# ============================================================
# BLOCK 11 — Run main scenarios
# ============================================================

scenario_results = {}

scenario_results["Optical"] = run_rf_cv(
    df=s2_model_df,
    features=OPTICAL_FEATURES,
    scenario_name="Optical"
)

scenario_results["SAR"] = run_rf_cv(
    df=s1_model_df,
    features=SAR_FEATURES,
    scenario_name="SAR"
)

FUSION_FEATURES = OPTICAL_FEATURES + SAR_FEATURES

scenario_results["Fusion"] = run_rf_cv(
    df=fusion_df,
    features=FUSION_FEATURES,
    scenario_name="Fusion"
)

if fusion_glcm_df is not None:
    FUSION_GLCM_FEATURES = OPTICAL_FEATURES + SAR_FEATURES + S2_GLCM_FEATURES

    scenario_results["Fusion_S2_GLCM"] = run_rf_cv(
        df=fusion_glcm_df,
        features=FUSION_GLCM_FEATURES,
        scenario_name="Fusion_S2_GLCM"
    )

print("Scenarios completed:")
print(list(scenario_results.keys()))

## 7. Compare model performance

This section summarises the main performance metrics used in the manuscript: accuracy, precision, recall, F1-score, ROC AUC and average precision. The copy-ready output is useful for checking consistency between the notebook and the manuscript tables.


In [ ]:
# ============================================================
# BLOCK 12 — Metrics comparison table
# ============================================================

metrics_df = pd.DataFrame([res["metrics"] for res in scenario_results.values()])
metrics_df = metrics_df[
    [
        "Scenario",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC AUC",
        "Average precision",
        "n",
        "n_features"
    ]
]

print("\nMODEL COMPARISON:")
display(metrics_df)

print("\nCopy-ready metrics:")
for _, row in metrics_df.iterrows():
    print(
        f"{row['Scenario']}: "
        f"Accuracy={row['Accuracy']:.4f}; "
        f"Precision={row['Precision']:.4f}; "
        f"Recall={row['Recall']:.4f}; "
        f"F1-score={row['F1-score']:.4f}; "
        f"ROC AUC={row['ROC AUC']:.4f}; "
        f"Average precision={row['Average precision']:.4f}; "
        f"n={int(row['n'])}; "
        f"n_features={int(row['n_features'])}"
    )

In [ ]:
# ============================================================
# BLOCK 13 — Print full summaries for all scenarios
# ============================================================

for scenario_name, result in scenario_results.items():
    print_scenario_summary(result)

## 8. Sample-level error transitions and ambiguity

The following analyses compare predictions at the individual-sample level. This is important because similar global metrics can hide different error directions, especially when one model reduces forest-to-coffee errors without reducing coffee-to-forest errors.


In [ ]:
# ============================================================
# BLOCK 14 — Error transition analysis among scenarios
# ============================================================

# This block compares sample-level predictions across scenarios.
# It requires that the scenarios share the same sample_id set.
# The common intersection is used automatically.

comparison_tables = []

for name, res in scenario_results.items():
    temp = res["results"][["sample_id", "class", "group", "predicted_class", "y_prob_coffee", "correct"]].copy()
    temp = temp.rename(columns={
        "predicted_class": f"pred_{name}",
        "y_prob_coffee": f"prob_{name}",
        "correct": f"correct_{name}"
    })
    comparison_tables.append(temp)

comparison_df = comparison_tables[0]

for temp in comparison_tables[1:]:
    comparison_df = comparison_df.merge(
        temp.drop(columns=["class", "group"]),
        on="sample_id",
        how="inner"
    )

print("Scenario comparison dataframe shape:", comparison_df.shape)
print("Common samples across all scenarios:", comparison_df["sample_id"].nunique())

display(comparison_df.head())

In [ ]:
# ============================================================
# BLOCK 15 — Fusion improvement and persistent errors
# ============================================================

def transition_summary(df, base_scenario, target_scenario):
    base_correct = f"correct_{base_scenario}"
    target_correct = f"correct_{target_scenario}"
    base_pred = f"pred_{base_scenario}"
    target_pred = f"pred_{target_scenario}"

    if base_correct not in df.columns or target_correct not in df.columns:
        print(f"[WARNING] Cannot compare {base_scenario} and {target_scenario}.")
        return None

    out = df.copy()

    def transition_label(row):
        if (not row[base_correct]) and row[target_correct]:
            return "Fixed_by_target"
        if row[base_correct] and (not row[target_correct]):
            return "New_error_in_target"
        if (not row[base_correct]) and (not row[target_correct]):
            return "Persistent_error"
        return "Correct_in_both"

    out["transition"] = out.apply(transition_label, axis=1)

    summary = (
        out.groupby("transition")
        .agg(
            n=("sample_id", "count"),
            mean_prob_target=(f"prob_{target_scenario}", "mean")
        )
        .reset_index()
        .sort_values("transition")
    )

    print("\n" + "-"*70)
    print(f"Transition: {base_scenario} → {target_scenario}")
    print("-"*70)
    display(summary)

    # Direction of errors in target scenario
    error_rows = out[out["transition"].isin(["Fixed_by_target", "New_error_in_target", "Persistent_error"])].copy()

    if not error_rows.empty:
        direction_summary = (
            error_rows.groupby(["transition", "class", target_pred])
            .size()
            .reset_index(name="n")
            .sort_values(["transition", "class", target_pred])
        )
        print("\nTransition by class and target prediction:")
        display(direction_summary)

    return out, summary

transition_opt_to_fusion = transition_summary(comparison_df, "Optical", "Fusion")
transition_sar_to_fusion = transition_summary(comparison_df, "SAR", "Fusion")

if "Fusion_S2_GLCM" in scenario_results:
    transition_fusion_to_glcm = transition_summary(comparison_df, "Fusion", "Fusion_S2_GLCM")

In [ ]:
# ============================================================
# BLOCK 16 — Probability-based ambiguity analysis
# ============================================================

def ambiguity_category(p):
    if p < 0.25:
        return "Strong forest signal"
    if p < 0.45:
        return "Forest-like"
    if p <= 0.55:
        return "Ambiguous"
    if p <= 0.75:
        return "Coffee-like"
    return "Strong coffee signal"

ambiguity_summaries = []

for name, res in scenario_results.items():
    temp = res["results"].copy()
    temp["ambiguity_class"] = temp["y_prob_coffee"].apply(ambiguity_category)

    summary = (
        temp.groupby(["group", "ambiguity_class"])
        .agg(n=("sample_id", "count"))
        .reset_index()
    )
    summary["Scenario"] = name
    ambiguity_summaries.append(summary)

ambiguity_df = pd.concat(ambiguity_summaries, ignore_index=True)

print("\nAMBIGUITY SUMMARY BY GROUP AND SCENARIO:")
display(ambiguity_df)

print("\nAmbiguity summary for coffee groups only:")
display(ambiguity_df[ambiguity_df["group"].str.startswith("coffee", na=False)])

## 9. Univariate separability of final predictors

This section evaluates the marginal separability of individual predictors using class means, robust effect-size summaries and AUC. These results should be interpreted as diagnostic evidence rather than as a replacement for the multivariate Random Forest models.


In [ ]:
# ============================================================
# BLOCK 17 — Univariate separability of final features
# ============================================================

def univariate_separability(df, features, scenario_name):
    data = df.copy()
    data = data[data["class"].isin(["coffee", "forest"])].copy()
    data = data.dropna(subset=features + ["class"]).copy()

    y = data["class"].map({"forest": 0, "coffee": 1}).values

    rows = []

    for f in features:
        x = data[f].values

        try:
            auc_raw = roc_auc_score(y, x)
            auc = max(auc_raw, 1 - auc_raw)
        except Exception:
            auc_raw = np.nan
            auc = np.nan

        coffee_vals = data.loc[data["class"] == "coffee", f].dropna()
        forest_vals = data.loc[data["class"] == "forest", f].dropna()

        mean_coffee = coffee_vals.mean()
        mean_forest = forest_vals.mean()
        diff_forest_minus_coffee = mean_forest - mean_coffee

        sd_pooled = np.sqrt(
            ((len(coffee_vals) - 1) * coffee_vals.var(ddof=1) + (len(forest_vals) - 1) * forest_vals.var(ddof=1)) /
            (len(coffee_vals) + len(forest_vals) - 2)
        )

        cohens_d = (mean_coffee - mean_forest) / sd_pooled if sd_pooled != 0 else np.nan

        try:
            _, p_value = mannwhitneyu(coffee_vals, forest_vals, alternative="two-sided")
        except Exception:
            p_value = np.nan

        rows.append({
            "Scenario": scenario_name,
            "Feature": f,
            "AUC": auc,
            "AUC_raw": auc_raw,
            "Cohens_d": cohens_d,
            "abs_Cohens_d": abs(cohens_d) if pd.notna(cohens_d) else np.nan,
            "p_value": p_value,
            "Mean_coffee": mean_coffee,
            "Mean_forest": mean_forest,
            "Diff_forest_minus_coffee": diff_forest_minus_coffee,
            "n_coffee": len(coffee_vals),
            "n_forest": len(forest_vals)
        })

    out = pd.DataFrame(rows)
    out = out.sort_values("AUC", ascending=False).reset_index(drop=True)
    return out

sep_optical_df = univariate_separability(s2_model_df, OPTICAL_FEATURES, "Optical")
sep_sar_df = univariate_separability(s1_model_df, SAR_FEATURES, "SAR")

print("\nOPTICAL UNIVARIATE SEPARABILITY:")
display(sep_optical_df)

print("\nSAR UNIVARIATE SEPARABILITY:")
display(sep_sar_df)

if fusion_glcm_df is not None:
    sep_glcm_df = univariate_separability(fusion_glcm_df, S2_GLCM_FEATURES, "S2_GLCM")
    print("\nS2 GLCM UNIVARIATE SEPARABILITY:")
    display(sep_glcm_df)

## 10. Export derived outputs

Only derived, non-geometric outputs are exported. These files are suitable for internal checking, manuscript tables or optional sharing when they contain no sensitive sample coordinates or private identifiers.


In [ ]:
# ============================================================
# BLOCK 18 — Export final outputs
# ============================================================

os.makedirs(output_dir, exist_ok=True)

metrics_path = os.path.join(output_dir, "final_model_metrics_optical_sar_fusion_glcm.csv")
comparison_path = os.path.join(output_dir, "final_sample_level_predictions_all_scenarios.csv")
sep_optical_path = os.path.join(output_dir, "final_univariate_separability_optical.csv")
sep_sar_path = os.path.join(output_dir, "final_univariate_separability_sar.csv")

metrics_df.to_csv(metrics_path, index=False)
comparison_df.to_csv(comparison_path, index=False)
sep_optical_df.to_csv(sep_optical_path, index=False)
sep_sar_df.to_csv(sep_sar_path, index=False)

print("Exported metrics to:", metrics_path)
print("Exported sample-level predictions to:", comparison_path)
print("Exported Optical separability to:", sep_optical_path)
print("Exported SAR separability to:", sep_sar_path)

if fusion_glcm_df is not None:
    sep_glcm_path = os.path.join(output_dir, "final_univariate_separability_s2_glcm.csv")
    sep_glcm_df.to_csv(sep_glcm_path, index=False)
    print("Exported S2 GLCM separability to:", sep_glcm_path)

## AI-use statement

Large language models were used as writing and coding support tools during the preparation of this notebook, specifically to assist with code organisation, documentation, readability and repository formatting. All scientific decisions, dataset interpretation, parameter choices and final analytical responsibility remain with the authors.
